# 유방암 데이터로 과제 2: 하이퍼파라미터 튜닝
## - GridSearchCV vs RandomSearchCV 비교
## - LogisticRegression의 최적 파라미터 찾기
## - 정확도와 AUC 모두 확인

In [63]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from sklearn.metrics import accuracy_score, roc_auc_score

In [28]:
# GridSearchCV

In [43]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test =  train_test_split (X,y,test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000, random_state=42) 
# max_iter=1000 한 건 solver가 수렴 못 해서 경고 뜨는 걸 방지

In [44]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100], 
    #C: 규제 세기 조절하는 하이퍼파라미터 (작을수록 규제 강함) 기본적으로는 C=1
    'penalty': ['l1','l2'],
    'solver':['liblinear','saga']}

In [45]:
grid_search = GridSearchCV(
    estimator = model,   # estimator :파라미터의 추정치(estimate)를 계산하는 규칙 또는 함수
    param_grid = param_grid,
    cv=5, # cv fold 갯수 / 5개로 나눴다
    scoring='accuracy', # 평가 지표
    n_jobs=-1,    # 병렬 처리 (모든 CPU 사용)
    verbose=2 # 진행사항 출력
)

In [89]:
grid_search.fit(X_train,y_train)

best_grid_model = grid_search.best_estimator_

train_acc = best_grid_model.score(X_train, y_train)
test_acc = best_grid_model.score(X_test, y_test)
GridSearch_auc = roc_auc_score(y_test, y_proba_grid)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-pack

In [53]:
print("="*60)
print(f"GridSearch 최적 파라미터: {grid_search.best_params_}")
print(f"GridSearch 최적 점수: {grid_search.best_score_:.3f}")
print("="*60)

GridSearch 최적 파라미터: {'C': 100, 'penalty': 'l1', 'solver': 'liblinear'}
GridSearch 최적 점수: 0.967
[CV] END ..............C=0.001, penalty=l1, solver=liblinear; total time=   0.0s
[CV] END ..............C=0.001, penalty=l2, solver=liblinear; total time=   0.0s
[CV] END ...................C=0.001, penalty=l2, solver=saga; total time=   0.1s
[CV] END ...............C=0.01, penalty=l1, solver=liblinear; total time=   0.0s
[CV] END ....................C=0.01, penalty=l1, solver=saga; total time=   0.1s
[CV] END .....................C=0.1, penalty=l1, solver=saga; total time=   0.1s
[CV] END .....................C=0.1, penalty=l2, solver=saga; total time=   0.1s
[CV] END .................C=10, penalty=l1, solver=liblinear; total time=   0.6s
[CV] END .................C=10, penalty=l1, solver=liblinear; total time=   0.2s
[CV] END ..C=73.20039418114051, penalty=l1, solver=liblinear; total time=   0.2s
[CV] END ..C=15.60286404424365, penalty=l1, solver=liblinear; total time=   0.6s
[CV] END ..C=6

In [67]:
print("="*60,"GridSearch","="*60)
print(f"GridSearch 최적 파라미터: {grid_search.best_params_}")
print(f"최적 점수: {grid_search.best_score_:.3f}")
print("="*60)
print(f"Train Accuracy(정확도): {train_acc:.3f}")
print(f"Test Accuracy(정확도): {test_acc:.3f}")
print("="*60)
print(f"AUC: {GridSearch_auc:.3f}")
print("="*60)

Train Accuracy(정확도): 0.989
Test Accuracy(정확도): 0.982
AUC: 0.996


In [34]:
# RandomSearchCV

In [54]:
results = pd.DataFrame(grid_search.cv_results_)

In [55]:
param_distributions={
    'C': uniform(0.001, 100),
    'penalty': ['l1','l2'],
    'solver': ['liblinear','saga']}

In [56]:
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=50,              # 50개 조합만 시도
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=2)

In [72]:
random_search.fit(X_train, y_train)

best_random_model = random_search.best_estimator_

train_acc = best_random_model.score(X_train, y_train)
test_acc = best_random_model.score(X_test, y_test)
RandomSearch_auc = roc_auc_score(y_test, y_proba_grid)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.12/site-pack

In [87]:
print("="*45,"RandomSearch","="*45)
print(f"최적 파라미터: {random_search.best_params_}")
print(f"최적 점수: {random_search.best_score_:.3f}")
print("="*104)
print(f"Train Accuracy(정확도): {random_train_acc:.3f}")
print(f"Test Accuracy(정확도): {random_test_acc:.3f}")
print("="*104)
print(f"AUC: {RandomSearch_auc:.3f}")
print("="*104)

============================================= RandomSearch =============================================
최적 파라미터: {'C': np.float64(73.20039418114051), 'penalty': 'l1', 'solver': 'liblinear'}
최적 점수: 0.969
Train Accuracy(정확도): 0.989
Test Accuracy(정확도): 0.982
AUC: 0.996
